# Lab 25: Linear Algebra and Optimization

This independent-study notebook accompanies **Chapter 25: Linear Algebra and Optimization**.

You will compute:

1. minimizers of positive definite quadratic functions;
2. gradient descent paths;
3. least squares and ridge regression solutions;
4. equality-constrained solutions using KKT systems;
5. Rayleigh quotient optimization.

Each section includes worked answers and similar practice questions.

## 1. Quadratic optimization

For
$$
f(x)=\frac12x^TQx-b^Tx,
$$
if $Q=Q^T\succ0$, the unique minimizer satisfies
$$
Qx_*=b.
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

### Worked example 1

Let
$$
Q=\begin{bmatrix}4&1\\1&3\end{bmatrix},\qquad b=\begin{bmatrix}1\\2\end{bmatrix}.
$$
Find the minimizer.

In [ ]:
Q = np.array([[4., 1.],
              [1., 3.]])
b = np.array([1., 2.])

eigvals = np.linalg.eigvalsh(Q)
x_star = np.linalg.solve(Q, b)
f_star = 0.5 * x_star @ Q @ x_star - b @ x_star

print("Eigenvalues of Q:", eigvals)
print("Minimizer x*:", x_star)
print("Minimum value:", f_star)

**Answer.** Since both eigenvalues of $Q$ are positive, $Q$ is positive definite. The minimizer is obtained by solving $Qx=b$.

### Similar practice 1

Let
$$
Q=\begin{bmatrix}3&1\\1&2\end{bmatrix},\qquad b=\begin{bmatrix}1\\4\end{bmatrix}.
$$
Compute the minimizer.

In [ ]:
Qp = np.array([[3., 1.],
               [1., 2.]])
bp = np.array([1., 4.])

xp = np.linalg.solve(Qp, bp)
print("Practice minimizer:", xp)

**Answer.**
$$
x_*=\left(-\frac25,\frac{11}{5}\right)^T.
$$

## 2. Gradient descent

Gradient descent is
$$
x_{k+1}=x_k-\alpha\nabla f(x_k).
$$
For a quadratic,
$$
\nabla f(x)=Qx-b.
$$

In [ ]:
def quadratic_value(Q, b, x):
    return 0.5 * x @ Q @ x - b @ x

def gradient_descent(Q, b, x0, alpha, steps):
    x = x0.astype(float).copy()
    history = [x.copy()]
    values = [quadratic_value(Q, b, x)]
    for _ in range(steps):
        grad = Q @ x - b
        x = x - alpha * grad
        history.append(x.copy())
        values.append(quadratic_value(Q, b, x))
    return np.array(history), np.array(values)

Q = np.array([[5., 1.],
              [1., 2.]])
b = np.array([1., 1.])
x0 = np.array([3., -2.])
alpha = 0.25

hist, vals = gradient_descent(Q, b, x0, alpha, 25)
print("Last iterate:", hist[-1])
print("True minimizer:", np.linalg.solve(Q, b))
print("Last five objective values:", vals[-5:])

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(vals, marker="o")
plt.xlabel("iteration")
plt.ylabel("objective value")
plt.title("Gradient descent objective values")
plt.grid(True)
plt.show()

### Similar practice 2

Try $\alpha=0.8$ for the same $Q$. Compare the behavior. Explain using the condition
$$
0<\alpha<\frac{2}{\lambda_{\max}(Q)}.
$$

In [ ]:
eigvals = np.linalg.eigvalsh(Q)
print("lambda_max =", eigvals[-1])
print("2/lambda_max =", 2/eigvals[-1])

hist_bad, vals_bad = gradient_descent(Q, b, x0, 0.8, 15)
print("Values for alpha=0.8:")
print(vals_bad)

**Answer.** Since $2/\lambda_{\max}$ is the largest stable threshold, a step size larger than this threshold causes instability or divergence.

## 3. Least squares

The least squares problem
$$
\min_x\|Ax-b\|_2^2
$$
has normal equations
$$
A^TAx=A^Tb.
$$

In [ ]:
A = np.array([[1., 0.],
              [1., 1.],
              [1., 2.]])
b = np.array([1., 2., 2.])

AtA = A.T @ A
Atb = A.T @ b
x_normal = np.linalg.solve(AtA, Atb)
x_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]

print("A^T A =")
print(AtA)
print("A^T b =", Atb)
print("Normal equation solution:", x_normal)
print("np.linalg.lstsq solution:", x_lstsq)

**Answer.** The two answers agree. The vector $Ax_*$ is the orthogonal projection of $b$ onto $\operatorname{Col}(A)$.

## 4. Ridge regression

Ridge regression solves
$$
\min_x \|Ax-b\|_2^2+\lambda\|x\|_2^2.
$$
The solution satisfies
$$
(A^TA+\lambda I)x=A^Tb.
$$

In [ ]:
A = np.array([[1., 1.00],
              [1., 1.01],
              [1., 0.99],
              [1., 1.02]])
b = np.array([2.0, 2.01, 1.99, 2.02])

print("Condition number of A^T A:", np.linalg.cond(A.T @ A))

for lam in [0, 0.001, 0.01, 0.1, 1.0]:
    if lam == 0:
        x = np.linalg.lstsq(A, b, rcond=None)[0]
    else:
        x = np.linalg.solve(A.T @ A + lam*np.eye(A.shape[1]), A.T @ b)
    print(f"lambda={lam:>5}: x={x}")

**Answer.** Larger $\lambda$ stabilizes the system by replacing $A^TA$ with $A^TA+\lambda I$, a positive definite matrix.

## 5. Equality constraints and KKT systems

Consider
$$
\min_x \frac12x^TQx-b^Tx
\quad \text{subject to}\quad Cx=d.
$$
The KKT system is
$$
\begin{bmatrix}
Q&C^T\\
C&0
\end{bmatrix}
\begin{bmatrix}
x\\
\lambda
\end{bmatrix}
=
\begin{bmatrix}
b\\
d
\end{bmatrix}.
$$

In [ ]:
Q = np.array([[2., 0.],
              [0., 2.]])
b = np.array([2., 0.])
C = np.array([[1., 1.]])
d = np.array([1.])

KKT = np.block([[Q, C.T],
                [C, np.zeros((C.shape[0], C.shape[0]))]])
rhs = np.concatenate([b, d])

sol = np.linalg.solve(KKT, rhs)
x_star = sol[:2]
lam_star = sol[2:]

print("KKT matrix =")
print(KKT)
print("x* =", x_star)
print("lambda* =", lam_star)
print("constraint Cx =", C @ x_star)

### Similar practice 3

Project $y=(2,-1)^T$ onto the line $x_1+x_2=1$ by solving a KKT system.

In [ ]:
y = np.array([2., -1.])
C = np.array([[1., 1.]])
d = np.array([1.])

I = np.eye(2)
KKT = np.block([[I, C.T],
                [C, np.zeros((1,1))]])
rhs = np.concatenate([y, d])

sol = np.linalg.solve(KKT, rhs)
x_proj = sol[:2]
lam = sol[2:]

print("Projection:", x_proj)
print("lambda:", lam)
print("Check x1+x2:", x_proj.sum())

**Answer.** The projected point is $(2,-1)^T$ shifted along the normal direction $(1,1)^T$ until it lies on $x_1+x_2=1$.

## 6. Rayleigh quotient

For a symmetric matrix $A$,
$$
R_A(x)=\frac{x^TAx}{x^Tx}.
$$
The minimum and maximum values of $R_A(x)$ are the smallest and largest eigenvalues of $A$.

In [ ]:
A = np.array([[2., 1.],
              [1., 4.]])

eigvals, eigvecs = np.linalg.eigh(A)

def rayleigh(A, x):
    return (x @ A @ x)/(x @ x)

samples = []
for theta in np.linspace(0, 2*np.pi, 200):
    x = np.array([np.cos(theta), np.sin(theta)])
    samples.append(rayleigh(A, x))

print("Eigenvalues:", eigvals)
print("Sample min/max Rayleigh:", min(samples), max(samples))

In [ ]:
plt.figure(figsize=(6,4))
theta = np.linspace(0, 2*np.pi, 200)
plt.plot(theta, samples)
plt.xlabel("theta")
plt.ylabel("Rayleigh quotient")
plt.title("Rayleigh quotient on the unit circle")
plt.grid(True)
plt.show()

## Final reflection

Write a short paragraph answering:

1. Why is solving $Qx=b$ an optimization method?
2. Why does least squares use $A^TA$?
3. How do eigenvalues control gradient descent?
4. What does a Lagrange multiplier measure in a KKT system?